# Guide Tasks
This notebook implements the 5 tasks provided by the project guide:
1. Download the required dataset for your project.
2. Load it to Pandas, check for missing values.
3. Drop the redundant columns.
4. Perform exploratory data analysis.
5. Compare 2 algorithms and visualize the result.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
sns.set_theme(style="whitegrid")

### Task 1 & 2: Load the Dataset and Check for Missing Values

In [ ]:
# Load the pre-processed dataset from the data folder
df = pd.read_csv('../data/processed/final_training_table.csv')

print("Dataset Shape:", df.shape)
print("\nMissing values per column:")
print(df.isnull().sum())

df.head()

### Task 3: Drop Redundant Columns
We will drop columns that are identifiers or metadata (e.g., `area_id`, `area_name`, `state_name`, etc.) which are not useful for model prediction.

In [ ]:
columns_to_drop = ['area_id', 'area_name', 'state_name', 'data_mode', 'data_source', 'data_year']

# Check which columns to drop actually exist in the dataframe to avoid errors
existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]

df_cleaned = df.drop(columns=existing_columns_to_drop)

print("Columns after dropping redundant ones:")
print(df_cleaned.columns.tolist())
df_cleaned.head()

### Task 4: Exploratory Data Analysis

In [ ]:
# 1. Distribution of the target variable (deprivation_score)
plt.figure(figsize=(8, 5))
sns.histplot(df_cleaned['deprivation_score'], kde=True, color='blue', bins=30)
plt.title('Distribution of Deprivation Score')
plt.xlabel('Deprivation Score')
plt.ylabel('Frequency')
plt.show()

# 2. Correlation Heatmap
plt.figure(figsize=(10, 8))
correlation_matrix = df_cleaned.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.show()

# 3. Scatter plot of a key feature vs target
plt.figure(figsize=(8, 5))
sns.scatterplot(x='nightlight_mean', y='deprivation_score', data=df_cleaned, alpha=0.6, color='purple')
plt.title('Nightlight Mean vs Deprivation Score')
plt.xlabel('Nightlight Mean')
plt.ylabel('Deprivation Score')
plt.show()

### Task 5: Compare 2 Algorithms and Visualize the Result
We will compare **Linear Regression** and **Random Forest Regressor** for predicting the `deprivation_score`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Prepare features (X) and target (y)
X = df_cleaned.drop(columns=['deprivation_score'])
y = df_cleaned['deprivation_score']

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Algorithm 1: Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_predictions = lr_model.predict(X_test)
lr_mse = mean_squared_error(y_test, lr_predictions)
lr_r2 = r2_score(y_test, lr_predictions)

# Algorithm 2: Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)
rf_mse = mean_squared_error(y_test, rf_predictions)
rf_r2 = r2_score(y_test, rf_predictions)

print("Linear Regression Results:")
print(f"MSE: {lr_mse:.4f}")
print(f"R-squared: {lr_r2:.4f}\n")

print("Random Forest Results:")
print(f"MSE: {rf_mse:.4f}")
print(f"R-squared: {rf_r2:.4f}")

In [ ]:
# Visualize the comparison: Actual vs Predicted for both models
plt.figure(figsize=(14, 6))

# Linear Regression Plot
plt.subplot(1, 2, 1)
plt.scatter(y_test, lr_predictions, alpha=0.5, color='orange')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title(f'Linear Regression\n$R^2$ = {lr_r2:.3f}')
plt.xlabel('Actual Deprivation Score')
plt.ylabel('Predicted Deprivation Score')

# Random Forest Plot
plt.subplot(1, 2, 2)
plt.scatter(y_test, rf_predictions, alpha=0.5, color='green')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title(f'Random Forest\n$R^2$ = {rf_r2:.3f}')
plt.xlabel('Actual Deprivation Score')
plt.ylabel('Predicted Deprivation Score')

plt.tight_layout()
plt.show()